In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 04 — Robustness under Data Perturbation

Menguji subsampling, noise sintetis, penghilangan dokumen, dan
perubahan periode waktu.


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
%cd {PROJECT_ROOT}

%pip install -q \
    "gensim==4.4.0" \
    sentence-transformers \
    pyarrow \
    openpyxl

In [ ]:
import torch
import gensim
import sentence_transformers

print("CUDA:", torch.cuda.is_available())
print("Gensim:", gensim.__version__)
print(
    "Sentence Transformers:",
    sentence_transformers.__version__,
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import numpy as np
import pandas as pd
from src.preprocessing import load_feature_bundle

bundle = load_feature_bundle(FEATURE_DIR)
raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
bow_normalized = bundle["bow_normalized"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

print("Documents:", len(raw_texts))
print("Vocabulary:", len(vocabulary))
print("Semantic shape:", semantic_embeddings.shape)
print("Document shape:", document_embeddings.shape)


In [ ]:
assert len(metadata_df) == len(raw_texts), (
    "Jumlah metadata tidak sama dengan jumlah feature."
)

In [ ]:
import json
import pandas as pd
import torch

from src.hmrnet import HMRNet, HMRNetConfig
from src.perturbation import (
    create_temporal_phases,
    perturb_texts,
    relative_performance_degradation,
    remove_documents,
)
from src.preprocessing import (
    PreprocessConfig,
    TextPreprocessor,
    build_feature_bundle,
)
from src.topic_metrics import evaluate_topics

metadata_df = pd.read_parquet(
    CHECKPOINTS_DIR / "dataset_metadata.parquet"
)
columns = json.loads(
    (CHECKPOINTS_DIR / "resolved_columns.json").read_text()
)

N_TOPICS = 20
EPOCHS = 40
TOP_N_WORDS = 10
SUBSAMPLE_PROPORTIONS = [0.25, 0.50, 0.75, 1.00]
NOISE_LEVELS = [0.05, 0.10, 0.20, 0.30]
NOISE_TYPES = [
    "token_deletion",
    "typo",
    "stopword_removal",
    "slang",
    "document_removal",
]


In [ ]:
preprocess_config = PreprocessConfig(
    min_df=2,
    max_df=0.95,
    max_features=min(12000, len(vocabulary)),
    document_dim=document_embeddings.shape[1],
    random_state=42,
)
preprocessor = TextPreprocessor(preprocess_config)

def train_and_evaluate(texts, seed=42):
    processed = preprocessor.transform(texts)
    mask = [bool(text.strip()) for text in processed]
    raw_valid = [text for text, valid in zip(texts, mask) if valid]
    processed_valid = [
        text for text, valid in zip(processed, mask) if valid
    ]
    features = build_feature_bundle(
        raw_texts=raw_valid,
        processed_texts=processed_valid,
        config=preprocess_config,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    model = HMRNet(
        HMRNetConfig(
            n_topics=N_TOPICS,
            epochs=EPOCHS,
            random_state=seed,
        )
    ).fit(
        bow=features["bow"],
        semantic_embeddings=features["semantic_embeddings"],
        document_embeddings=features["document_embeddings"],
        vocabulary=features["vocabulary"],
        verbose=False,
    )
    metrics = evaluate_topics(
        "HMRNet-Full",
        model.get_topics(TOP_N_WORDS),
        processed_valid,
        top_n=TOP_N_WORDS,
    )
    return metrics, model, features


## 4.1 Subsampling robustness

In [ ]:
subsampling_rows = []

for proportion in SUBSAMPLE_PROPORTIONS:
    sample = metadata_df.sample(
        frac=proportion,
        random_state=42,
    )
    texts = sample[columns["text_column"]].astype(str).tolist()
    metrics, _, _ = train_and_evaluate(texts)
    subsampling_rows.append(
        {
            "Proportion": proportion,
            "Number_of_Documents": len(texts),
            **metrics,
        }
    )

subsampling_results = pd.DataFrame(subsampling_rows)
reference_cv = float(
    subsampling_results.loc[
        subsampling_results["Proportion"] == 1.0, "C_v"
    ].iloc[0]
)
subsampling_results["RPD_C_v"] = subsampling_results["C_v"].apply(
    lambda value: relative_performance_degradation(
        reference_cv, value, higher_is_better=True
    )
)
display(subsampling_results.round(4))


## 4.2 Noise robustness

In [ ]:
original_texts = metadata_df[
    columns["text_column"]
].astype(str).tolist()

baseline_metrics, _, _ = train_and_evaluate(original_texts)
baseline_cv = baseline_metrics["C_v"]
noise_rows = []

for noise_type in NOISE_TYPES:
    for noise_level in NOISE_LEVELS:
        if noise_type == "document_removal":
            noisy_texts = remove_documents(
                original_texts,
                noise_level=noise_level,
                random_state=42,
            )
        else:
            noisy_texts = perturb_texts(
                original_texts,
                noise_type=noise_type,
                noise_level=noise_level,
                random_state=42,
            )

        metrics, _, _ = train_and_evaluate(noisy_texts)
        noise_rows.append(
            {
                "Noise_Type": noise_type,
                "Noise_Level": noise_level,
                "Number_of_Documents": len(noisy_texts),
                **metrics,
                "RPD_C_v": relative_performance_degradation(
                    baseline_cv,
                    metrics["C_v"],
                    higher_is_better=True,
                ),
            }
        )

noise_results = pd.DataFrame(noise_rows)
display(noise_results.round(4))


## 4.3 Temporal robustness

In [ ]:
temporal_results = pd.DataFrame()

if columns.get("date_column"):
    temporal_df = create_temporal_phases(
        metadata_df,
        date_column=columns["date_column"],
    )
    temporal_rows = []
    for phase, group in temporal_df.groupby(
        "temporal_phase", dropna=True
    ):
        texts = group[
            columns["text_column"]
        ].astype(str).tolist()
        metrics, _, _ = train_and_evaluate(texts)
        temporal_rows.append(
            {
                "Temporal_Phase": phase,
                "Number_of_Documents": len(texts),
                **metrics,
            }
        )
    temporal_results = pd.DataFrame(temporal_rows)
    display(temporal_results.round(4))
else:
    print(
        "Kolom tanggal tidak tersedia; temporal robustness dilewati."
    )


In [ ]:
subsampling_results.to_csv(
    TABLES_DIR / "experiment_04_subsampling.csv",
    index=False,
)
noise_results.to_csv(
    TABLES_DIR / "experiment_04_noise.csv",
    index=False,
)
if len(temporal_results):
    temporal_results.to_csv(
        TABLES_DIR / "experiment_04_temporal.csv",
        index=False,
    )


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(
    subsampling_results["Proportion"] * 100,
    subsampling_results["C_v"],
    marker="o",
)
plt.xlabel("Data proportion (%)")
plt.ylabel("C_v coherence")
plt.title("HMRNet Subsampling Robustness")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_04_subsampling.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
